# 04 — Join Final + DuckDB

**Objetivo:** integrar todos os datasets no banco DuckDB, executar as **queries SQL obrigatórias** da disciplina (Q1–Q4) e persistir o dataset analítico final em Parquet.

Este é o único notebook que abre o arquivo `enem.duckdb` com acesso de escrita — todos os outros notebooks usam DuckDB in-memory ou leem o Parquet diretamente para evitar conflitos de lock.

## Estratégia de join

O join ENEM × Atlas usa uma estratégia de **fallback em dois níveis**:

```
1º) atlas_municipal (code_muni + ano)  →  dados municipais (quando disponíveis)
2º) atlas_uf        (sg_uf + ano)      →  dados estaduais (fallback atual)
```

Implementado com `COALESCE(am.idhm, auf.idhm)` em cada indicador.  
A coluna `nivel_atlas` registra qual nível foi usado para cada linha (`'municipal'` ou `'estadual'`).

> **Estado atual:** `atlas_municipal` está vazio — 100% das linhas usam o fallback estadual. Quando dados municipais forem obtidos e `atlas_municipal.parquet` for populado, reexecutar este notebook resolve automaticamente.

## Fluxo de dados
```
lookup_municipios.parquet  ──┐
enem_*.parquet (13 anos)    ──┤  DuckDB  →  dataset_analitico  →  Parquet + DuckDB
atlas_uf.parquet            ──┤
atlas_municipal.parquet     ──┘
```

## 1. Carregamento dos datasets no DuckDB

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

DUCKDB_PATH = '../data/enem.duckdb'
con = duckdb.connect(DUCKDB_PATH)

con.execute("DROP TABLE IF EXISTS lookup")
con.execute("CREATE TABLE lookup AS SELECT * FROM read_parquet('../data/processed/lookup/lookup_municipios.parquet')")

con.execute("DROP TABLE IF EXISTS enem")
con.execute("CREATE TABLE enem AS SELECT * FROM read_parquet('../data/processed/enem/enem_*.parquet', union_by_name=true)")

# Atlas estadual (27 UFs × anos) — sempre disponível
con.execute("DROP TABLE IF EXISTS atlas_uf")
con.execute("CREATE TABLE atlas_uf AS SELECT * FROM read_parquet('../data/processed/atlas/atlas_uf.parquet')")

# Atlas municipal (por enquanto vazio — preenchido quando tiver dados municipais)
con.execute("DROP TABLE IF EXISTS atlas_municipal")
con.execute("CREATE TABLE atlas_municipal AS SELECT * FROM read_parquet('../data/processed/atlas/atlas_municipal.parquet')")

print('Tabelas:', con.execute('SHOW TABLES').fetchall())


Tabelas: [('atlas_municipal',), ('atlas_uf',), ('dataset_analitico',), ('enem',), ('lookup',)]


## Q1 — Join principal ENEM × Atlas

**Query obrigatória Q1** da disciplina: integração dos microdados do ENEM com os indicadores socioeconômicos do Atlas Brasil.

O join usa `COALESCE` para priorizar dados municipais quando disponíveis, com fallback para estadual. O `CAST` é necessário porque todas as colunas dos Parquets ENEM são `TEXT` (padronização para compatibilidade entre anos — ver notebook 02).

A coluna `nivel_atlas` permite rastrear quantos candidatos têm dado municipal vs estadual — útil para documentar a limitação no artigo.

In [2]:
con.execute('DROP TABLE IF EXISTS dataset_analitico')
con.execute("""
    CREATE TABLE dataset_analitico AS
    SELECT
        e.NU_ANO,
        e.CO_MUNICIPIO_ESC,
        e.SG_UF_ESC,
        e.TP_SEXO, e.TP_COR_RACA, e.TP_ESCOLA,
        e.Q001, e.Q002, e.Q006, e.Q006_HARM, e.REGIAO,
        e.NU_NOTA_CN, e.NU_NOTA_CH, e.NU_NOTA_LC,
        e.NU_NOTA_MT, e.NU_NOTA_REDACAO,
        e.FAIXA_CN, e.FAIXA_CH, e.FAIXA_LC, e.FAIXA_MT, e.FAIXA_REDACAO,
        -- Questões socioeconômicas estendidas (disponíveis 2019-2024; NULL para 2012-2018)
        e.Q_OCUP_PAI, e.Q_OCUP_MAE, e.Q_N_PESSOAS,
        e.Q_EMPREGADA, e.Q_CARRO, e.Q_MOTO, e.Q_GELADEIRA,
        e.Q_LAVADORA, e.Q_TV, e.Q_COMPUTADOR, e.Q_INTERNET,
        e.Q_CELULAR, e.Q_TIPO_ESCOLA_EM, e.Q_BOLSA_FAM,
        -- Prioriza dado municipal; fallback para estadual
        COALESCE(am.idhm,              auf.idhm)              AS idhm,
        COALESCE(am.idhm_educacao,     auf.idhm_educacao)     AS idhm_educacao,
        COALESCE(am.idhm_renda,        auf.idhm_renda)        AS idhm_renda,
        COALESCE(am.idhm_longevidade,  auf.idhm_longevidade)  AS idhm_longevidade,
        COALESCE(am.renda_percapita,   auf.renda_percapita)   AS renda_percapita,
        COALESCE(am.tx_analfabetismo,  auf.tx_analfabetismo)  AS tx_analfabetismo,
        COALESCE(am.tx_envelhecimento, auf.tx_envelhecimento) AS tx_envelhecimento,
        COALESCE(am.esperanca_vida,    auf.esperanca_vida)    AS esperanca_vida,
        COALESCE(am.mortalidade_infantil, auf.mortalidade_infantil) AS mortalidade_infantil,
        COALESCE(am.idhm_branco,  auf.idhm_branco)  AS idhm_branco,
        COALESCE(am.idhm_negro,   auf.idhm_negro)   AS idhm_negro,
        COALESCE(am.idhm_homem,   auf.idhm_homem)   AS idhm_homem,
        COALESCE(am.idhm_mulher,  auf.idhm_mulher)  AS idhm_mulher,
        COALESCE(am.idhm_rural,   auf.idhm_rural)   AS idhm_rural,
        COALESCE(am.idhm_urbano,  auf.idhm_urbano)  AS idhm_urbano,
        -- Indica qual nível foi usado
        CASE WHEN am.idhm IS NOT NULL THEN 'municipal' ELSE 'estadual' END AS nivel_atlas
    FROM enem e
    LEFT JOIN atlas_municipal am
        ON CAST(e.CO_MUNICIPIO_ESC AS INTEGER) = am.code_muni
       AND CAST(e.NU_ANO AS INTEGER)           = am.ano
    LEFT JOIN atlas_uf auf
        ON e.SG_UF_ESC = auf.sg_uf
       AND CAST(e.NU_ANO AS INTEGER) = auf.ano
""")
total = con.execute('SELECT COUNT(*) FROM dataset_analitico').fetchone()[0]
print(f'dataset_analitico: {total:,} linhas')

# Mostra proporção municipal vs estadual
print(con.execute("""
    SELECT nivel_atlas, COUNT(*) AS n,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER(), 2) AS pct
    FROM dataset_analitico GROUP BY nivel_atlas
""").df().to_string(index=False))


dataset_analitico: 51,321,197 linhas
nivel_atlas        n   pct
   estadual 51321197 100.0


## Q2 — Nota média por município e ano

**Query obrigatória Q2:** agrega o desempenho médio por município (`CO_MUNICIPIO_ESC`) e ano, mostrando os municípios com maior nota média em Matemática.

Útil para identificar outliers geográficos — municípios com desempenho acima do esperado para seu estado.

In [3]:
con.execute("""
    SELECT CO_MUNICIPIO_ESC, NU_ANO,
           ROUND(AVG(CAST(NU_NOTA_MT AS DOUBLE)), 2)      AS media_MT,
           ROUND(AVG(CAST(NU_NOTA_REDACAO AS DOUBLE)), 2) AS media_REDACAO,
           COUNT(*) AS candidatos
    FROM dataset_analitico
    GROUP BY CO_MUNICIPIO_ESC, NU_ANO
    ORDER BY media_MT DESC LIMIT 10
""").df()


,CO_MUNICIPIO_ESC,NU_ANO,media_MT,media_REDACAO,candidatos
0,4313466,2020,836.10,720.00,1
1,4117297,2020,823.20,600.00,1
2,3143609,2021,807.10,940.00,1
3,3117207,2022,794.30,660.00,1
4,4305850,2020,787.40,780.00,1
5,3107000,2020,784.20,640.00,1
6,3555901,2023,779.10,920.00,1
7,4322103,2023,759.60,930.00,2
8,3546108,2020,738.00,600.00,1
9,3515806,2023,730.37,886.67,3


## Q3 — Distribuição de faixas de nota por macrorregião

**Query obrigatória Q3:** mostra o percentual de candidatos em cada faixa de nota (`<400` a `>800`) por macrorregião, usando window function `SUM() OVER (PARTITION BY REGIAO)` para calcular percentuais intra-regionais.

Esta query gera os dados da hipótese **H1** — evidência quantitativa da disparidade regional.

In [4]:
con.execute("""
    SELECT REGIAO, FAIXA_MT, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY REGIAO), 2) AS pct
    FROM dataset_analitico
    WHERE FAIXA_MT IS NOT NULL AND REGIAO IS NOT NULL
    GROUP BY REGIAO, FAIXA_MT
    ORDER BY REGIAO, FAIXA_MT
""").df()


,REGIAO,FAIXA_MT,n,pct
0,Centro-Oeste,400-500,466017,36.93
1,Centro-Oeste,500-600,318277,25.22
2,Centro-Oeste,600-700,179373,14.21
3,Centro-Oeste,700-800,65053,5.15
4,Centro-Oeste,<400,217895,17.27
5,Centro-Oeste,>800,15404,1.22
6,Nordeste,400-500,1718436,40.39
7,Nordeste,500-600,969997,22.80
8,Nordeste,600-700,490374,11.53
9,Nordeste,700-800,174074,4.09


## Q4 — Candidatos por ano após filtro de presença

**Query obrigatória Q4:** contagem de candidatos válidos por ano após aplicação do filtro de presença (ver notebook 02 para o volume removido por ano).

Nota: o ano de 2020 tem queda acentuada (~2,6M vs média de ~4,5M) devido ao COVID-19 — deve ser discutido no artigo como quebra estrutural na série temporal.

In [5]:
con.execute("""
    SELECT NU_ANO, COUNT(*) AS candidatos
    FROM dataset_analitico
    GROUP BY NU_ANO ORDER BY NU_ANO
""").df()


,NU_ANO,candidatos
0,2012,4079886
1,2013,5007934
2,2014,5947909
3,2015,5604905
4,2016,5818264
5,2017,4426692
6,2018,3893729
7,2019,3701910
8,2020,2588681
9,2021,2238107


## Validação da cobertura do join

Verifica o percentual de linhas com `idhm` preenchido após o join.

**Meta:** cobertura ≥ 85%. Com dados apenas estaduais, a cobertura é de ~27,7% (candidatos cujo estado tem dado Atlas no ano correspondente). Quando dados municipais forem obtidos, espera-se cobertura >85%.

In [6]:
cob = con.execute("""
    SELECT
        ROUND(100.0 * AVG(CASE WHEN idhm IS NOT NULL THEN 1.0 ELSE 0.0 END), 2) AS cob_idhm_pct,
        ROUND(100.0 * AVG(CASE WHEN idhm_negro IS NOT NULL THEN 1.0 ELSE 0.0 END), 2) AS cob_idhm_negro_pct
    FROM dataset_analitico
""").df()
print(cob)
print('Meta: cob_idhm_pct >= 85%')


   cob_idhm_pct  cob_idhm_negro_pct
0         27.73               27.73
Meta: cob_idhm_pct >= 85%


## Exportação do dataset analítico

Salva o dataset final em dois formatos:
- **Parquet:** `data/processed/dataset_analitico.parquet` (~777 MB) — para uso nos notebooks 05 e 06 sem depender do DuckDB
- **Tabela DuckDB:** persistida no `enem.duckdb` para queries SQL ad hoc

> O banco `enem.duckdb` contém ao final 5 tabelas: `lookup`, `enem`, `atlas_uf`, `atlas_municipal`, `dataset_analitico`.

In [7]:
con.execute("""
    COPY dataset_analitico TO '../data/processed/dataset_analitico.parquet' (FORMAT PARQUET)
""")
print('Salvo: data/processed/dataset_analitico.parquet')
# Mantém con aberto — notebook 05 usa read_only


Salvo: data/processed/dataset_analitico.parquet
